# SuperKernel 融合加速原理与实践

## 前置要求

学习本节前，建议你已了解 Ascend C 算子的基本开发流程，并完成前序课程《Aclgraph入图编译与运行》，了解 ACLGraph 的捕获与重放机制，以及通过 torch.compile(...,backend="npugraph_ex") 使能 ACLGraph 的基本方法。

## 学习目标

完成本节学习后，你应该能够：

- 理解 SuperKernel 的基本原理，了解其性能收益的主要来源；
- 掌握 ACLGraph 模式下 SuperKernel 的使能方法，并能通过使能SuperKernel来提升模型性能；
- 掌握 ACLGraph 模式下自定义算子适配接入 SuperKernel 的实现方式。

## 环境依赖

本节代码支持如下产品型号：

- Atlas A3 训练系列产品 / Atlas A3 推理系列产品；
- Atlas A2 训练系列产品 / Atlas A2 推理系列产品。

本节代码支持的cann版本：9.1.0 及以上

运行本节在线执行样例，需要当前环境已准备好以下软件：

- 已安装与硬件和驱动配套的 CANN 开发环境，并完成环境变量初始化；
- 完成昇腾 PyTorch 适配，并保证 `torch`、`torch_npu` 与当前 CANN、驱动、固件、Python 版本配套；安装与配套关系可参考 [Ascend Extension for PyTorch 安装](https://www.hiascend.com/document/detail/zh/Pytorch/latest/configandinstg/instg/docs/zh/installation_guide/installation_via_binary_package.md)。

运行后续在线执行样例前，请先执行下面的环境初始化单元。

In [ ]:
import os
import shlex
import subprocess
from pathlib import Path

CANN_SET_ENV = Path(os.environ.get("CANN_SET_ENV", "/usr/local/Ascend/cann/set_env.sh"))
if not CANN_SET_ENV.is_file():
    raise FileNotFoundError(
        "未找到 CANN set_env.sh，请设置 CANN_SET_ENV。"
    )

# 将 CANN 环境导入当前 Jupyter 进程，后续启动的终端命令会继承这些变量。
result = subprocess.run(
    ["bash", "-lc", f"source {shlex.quote(str(CANN_SET_ENV))} && env"],
    capture_output=True,
    text=True,
    check=True,
)
for line in result.stdout.splitlines():
    if "=" in line and not line.startswith(("#", " ")):
        key, value = line.split("=", 1)
        os.environ[key] = value

os.environ["CANN_SET_ENV"] = str(CANN_SET_ENV)
os.environ.setdefault("ASC_ARCH", "dav-2201")
os.environ["ASCEND_DEVICE_ID"] = "1"

print(f"CANN 环境脚本: {CANN_SET_ENV}")
print(f"Ascend C 架构: {os.environ['ASC_ARCH']}")
print(f"NPU 设备 ID: {os.environ['ASCEND_DEVICE_ID']}")
print("环境初始化完成。")


## 1. SuperKernel 介绍

SuperKernel是一种算子二进制融合技术。与源码融合不同，它聚焦于内核函数（Kernel）的二进制调度方案优化，在已编译的二进制代码基础上融合创建一个超级Kernel函数（简称SuperKernel），以子函数调用的方式整合多个内核函数，从而优化计算任务、提升性能和资源利用率。

与单算子下发相比，SuperKernel技术能够优化任务调度的等待时间和调度开销，并可利用Task间隙资源进一步优化算子头开销。


在模型执行前，SuperKernel 会分析用户标定范围内的 Kernel Task、执行顺序和依赖关系，识别支持SuperKernel融合的算子，并生成一个可统一启动的 SuperKernel Task。原来的 Kernel Task 不会消失，而是作为 SuperKernel 内部的 SubTask 执行。

<img src="./images/04_super_kernel/04_super_kernel_superkernel_before_after.png" alt="SuperKernel 融合前后对比" width="900" />

融合前，硬件需要分别调度多个 Task；融合后，硬件只需启动一次 SuperKernel Task，多个 SubTask 的执行在 SuperKernel 内部完成。

SuperKernel 的优化主要体现在以下几个方面：

- **减少 Task 结束后的Cache Flush开销。** 每个Task结束后会执行Cache Flush操作，将所有修改的Cache内容刷新出去，其中包含一些不必要的栈空间。
- **减少 Task 调度等待。** 多个独立 Task 之间通常存在硬件调度间隙。通过一次 SuperKernel 调度统一承接多个 Task，可以压缩 Task 逐个调度带来的等待时间。
- **减少 Task 启动开销。** 原本需要多次启动的 Task，可以通过一次 SuperKernel Task 启动统一完成，从而减少多次启动带来的额外开销。

<img src="./images/04_super_kernel/superkernel_performance_gain_model.png" alt="SuperKernel 性能收益模型" width="600" />

### 1.1 性能收益与分析案例

在开源 vLLM Ascend 的模型推理场景中，SuperKernel 可在不改变模型计算语义的前提下提供开箱即用的融合优化。对于存在连续可融合任务的网络片段，SuperKernel 可以减少任务调度和任务启动开销，从而改善模型推理性能。实际收益与模型结构、并发配置、输入输出长度以及软硬件版本有关，应以对应场景的实测数据为准。

| 模型 | 指标 | 使能 ACLGraph 性能 | 使能 SuperKernel 性能 |
|---|---|---:|---:|
| DeepSeek-R1-w4a8-mtp-QuaRot | TPOT（ms） | 45.3 ms | 41.7 ms |
| Qwen3-235B-A22B | TPOT（ms） | 46.6 ms | 45.4 ms |

注：TPOT 表示每个输出 Token 的平均生成时间，数值越低表示推理性能越好。

为了直观说明 SuperKernel 的性能收益，下面以一个连续执行 50 层的网络片段为例，观察 SuperKernel 对单层设备侧任务形态和执行耗时的影响。该网络每层包含 4 个主链路算子，并在末尾调用一次 clearops；输入形状为 `int8[9216, 7168]`，50 层共享同一套权重：

网络的核心结构如下，完整脚本位于 `src/06.04_super_kernel/performance_case/`：

```python
class Layer(torch.nn.Module):
    def forward(self, x):
        x = self.grouped_matmul_pre(x)
        x = self.dequant_swiglu_quant(x)
        x = self.grouped_matmul_after(x)
        x = self.dynamic_quant(x)
        x = clearops(x)
        return x


class Network(torch.nn.Module):
    def forward(self, x):
        for layer in self.layers:  # 连续执行 50 层
            x = layer(x)
        return x
```

两组实验使用相同的网络、输入和参数，并自动安装 ACLGraph clearops。对照组使能ACLGraph；实验组在此基础上使能SuperKernel，并采用逐层融合的方式。

以下性能分析基于已采集的 ACLGraph 和 ACLGraph+SK Profiling 数据。两组数据均由 msprof 导出，设备侧任务信息记录在对应 Profiling 结果的 `mindstudio_profiler_output` 目录下的 `op_summary_*.csv` 中。下面摘取第 0 层相关记录中的任务名称、任务类型、开始时间和执行时长进行比较。

ACLGraph 模式下，第 0 层包含 4 个主链路算子和 1 个 clearops 任务：

| OP Name | OP Type | Task Start Time(us) | Task Duration(us) |
|---|---|---:|---:|
| aclnnGroupedMatmulV5_GroupedMatmul_GroupedMatmul | GroupedMatmul | 1788186149743559.115 | 51.360 |
| DequantSwigluQuant | DequantSwigluQuant | 1788186149743611.235 | 7.660 |
| aclnnGroupedMatmulV5_GroupedMatmul_GroupedMatmul | GroupedMatmul | 1788186149743620.275 | 36.620 |
| aclnnDynamicQuantV2_DynamicQuant_DynamicQuant | DynamicQuant | 1788186149743657.875 | 358.740 |
| clear_ops | clear_ops | 1788186149744017.595 | 30.720 |

ACLGraph+SK 模式下，这一层被融合为 1 个 SuperKernel 任务。任务名称中可以看出SuperKernel融合范围的首算子是 `GroupedMatmul` ，尾算子是 `clear_ops` ：

| OP Name | OP Type | Task Start Time(us) | Task Duration(us) |
|---|---|---:|---:|
| sk_0_sk_test_layer_0_<br>start_static_kernel_GroupedMatmul_<br>513b506510b0747fe531659e08d025502f948632ff140daabec28a6556328687_<br>2482379_d0_end_clear_ops | SuperKernel | 1788186479100637.715 | 474.800 |

为保证开启 SK 前后的统计边界一致，设备侧耗时统一按下面的通用方式计算。当统计区间内只有一个 SuperKernel 任务时，首任务和末任务相同，因此计算结果就是该 SuperKernel 的 Task Duration。

```text
设备侧耗时 = 末任务 Task Start Time - 首任务 Task Start Time + 末任务 Task Duration
```

在本次采集结果中，SuperKernel 将第 0 层的 5 个设备侧任务融合为 1 个任务。未开启 SK 时单层耗时为 489.200 us，开启 SK 后单层耗时为 474.800 us，单层耗时减少约 14.4 us。如需在当前环境中重新运行该用例并采集 ACLGraph 与 ACLGraph+SK 的 Profiling 数据，可在终端执行以下命令。


```bash
cd src/06.04_super_kernel/performance_case
bash run_case.sh --mode static --device-id 3 --warmup 1
bash run_case.sh --mode sk --device-id 3 --warmup 1
```

## 2. 自定义算子适配 SuperKernel

要使模型中的算子参与 SuperKernel 融合优化，除在 ACLGraph 模式下开启 SuperKernel 外，算子本身还需要具备 SuperKernel 适配能力。不同接入方式的适配责任不同：对于 aclnn 算子，相关适配通常已由框架或算子实现侧完成，用户只需要在 ACLGraph 模式中开启 SuperKernel，并按需标定融合范围；对于自定义 <<<>>> 算子，开发者还需要实现可供 SuperKernel 调用的 __sk__ 子函数，并通过 SK_BIND 将其与普通 Kernel 入口绑定。

自定义算子参与 SuperKernel 融合时，除完成入图和使能外，还需要关注全核同步、核数启动比例、Cache 一致性等约束。具体要求可参考 [算子适配说明](https://asc.gitcode.com/guide/programming_guide/advanced_programming/super_kernel/operator_adaptation.html)。

本节讲解 ACLGraph+SK 场景下自定义 `<<<>>>` 算子的适配流程。

整体流程如下：

1. 实现普通 Ascend C Kernel，供 `add_custom<<<...>>>` 正常 launch。
2. 实现对应的 `__sk__` 子函数，供 SuperKernel 内部作为子函数调用。
3. 使用 `SK_BIND` 绑定普通 Kernel 入口和 SK 子函数。
4. 使用 pybind 暴露 Python 扩展接口。
5. 使用 `torch.library` 注册 PyTorch 算子，使模型中可以通过 `torch.ops` 调用。
6. 使用 `torch.compile(..., backend="npugraph_ex")` 开启静态 Kernel 编译和 SuperKernel 融合。

#### 2.1 实现 SK 子函数与绑定

自定义算子的普通 Kernel 源码位于 `src/06.04_super_kernel/custom_launch_sk/add_custom_kernel_base.asc`。其中，`add_custom` 是通过 `<<<>>>` 启动的设备侧 Kernel 入口，`add_custom_impl` 负责在当前 NPU Stream 上调用该入口；二者共同构成算子在普通执行路径中的实现。

要使该算子能够参与 SuperKernel 融合，需要在同一份算子源码中额外提供 SuperKernel 子函数入口。子函数使用 `__sk__` 修饰，并接收两类参数：一类是算子输入、输出等参数，另一类是 `sk::SkSystemArgs`。算子原有的计算逻辑仍可复用，例如继续创建 `KernelAdd` 并调用 `Init`、`Process`；但启动核数和核索引应通过 `sysArgs->SkGetNumBlocks()`、`sysArgs->SkGetBlockIdx()` 获取，使其适配 SuperKernel 的执行方式。

完成 SK 子函数后，使用 `SK_BIND` 将普通 Kernel 入口与对应的 SK 子函数关联。例如，`SK_BIND(add_custom, 4, add_custom_sk<0>, ..., add_custom_sk<3>)` 以 `add_custom` 作为普通 Kernel 入口，注册该算子可供 SuperKernel 调用的 `add_custom_sk` 子函数变体。完成绑定后，编译器在生成 SuperKernel 时即可识别该映射，并在融合后的 SuperKernel 内部调用对应子函数。

In [ ]:
%%bash
set -eo pipefail

SRC_DIR=src/06.04_super_kernel/custom_launch_sk
SOURCE_DIR=source/06.04_super_kernel/custom_launch_sk
BASE_FILE=${SRC_DIR}/add_custom_kernel_base.asc
TARGET_FILE=${SOURCE_DIR}/add_custom_kernel.asc

if [ ! -f "${BASE_FILE}" ]; then
    echo "未找到预置普通 Kernel 源码: ${BASE_FILE}" >&2
    exit 1
fi

mkdir -p "${SOURCE_DIR}"
cp "${BASE_FILE}" "${TARGET_FILE}"

cat >> "${TARGET_FILE}" <<'CPP'

#include "super_kernel/super_kernel.h"

struct AddCustomArgs {
    GM_ADDR x;
    GM_ADDR y;
    GM_ADDR z;
    uint32_t totalLength;
};

template <uint32_t splitNum>
__sk__ __vector__ void add_custom_sk(
    const AddCustomArgs *args, sk::SkSystemArgs *sysArgs)
{
    (void)splitNum;

    KernelAdd op;
    op.Init(args->x, args->y, args->z, args->totalLength, TILE_NUM,
            sysArgs->SkGetNumBlocks(), sysArgs->SkGetBlockIdx());
    op.Process();
}

SK_BIND(add_custom, 4,
        add_custom_sk<0>, add_custom_sk<1>,
        add_custom_sk<2>, add_custom_sk<3>);
CPP

echo "已生成 ${TARGET_FILE}"


#### 2.2 通过 pybind 和 torch.library 接入 PyTorch

完成 Kernel 和 SuperKernel 绑定后，还需要把自定义算子接入 PyTorch。pybind 层负责把 C++/Ascend C host 函数暴露成 Python 扩展；`torch.library` 负责把该扩展注册成 PyTorch 算子，使模型中可以通过 `torch.ops` 调用，并让 `torch.compile` 能进行图捕获和 Meta 推导。

host 包装函数创建输出 Tensor，并使用当前 PyTorch NPU stream 调用 `.asc` 中提供的 `add_custom_impl`，普通 `<<<>>>` launch 逻辑仍保留在 Ascend C 源码内。


In [ ]:
%%writefile source/06.04_super_kernel/custom_launch_sk/add_custom_pytorch.cpp
#include <cstdint>
#include <torch/extension.h>
#include "torch_npu/csrc/core/npu/NPUStream.h"

namespace {
constexpr int64_t ROW_NUM = 8;
constexpr int64_t COL_NUM = 2048;
}  // namespace

extern "C" void add_custom_impl(void *aclStream,
                                uint8_t *x,
                                uint8_t *y,
                                uint8_t *z);

namespace ascendc_ops {
at::Tensor run_add_custom(const at::Tensor &x, const at::Tensor &y)
{
    TORCH_CHECK(x.sizes() == y.sizes(), "x and y must have the same shape.");
    TORCH_CHECK(x.dim() == 2 && x.size(0) == ROW_NUM && x.size(1) == COL_NUM,
                "add_custom only supports shape [8, 2048] in this sample.");
    TORCH_CHECK(x.scalar_type() == at::ScalarType::Float &&
                    y.scalar_type() == at::ScalarType::Float,
                "add_custom only supports float32 tensors in this sample.");

    at::Tensor z = at::empty_like(x);
    auto aclStream = c10_npu::getCurrentNPUStream().stream(false);

    add_custom_impl(aclStream,
                    reinterpret_cast<uint8_t *>(x.data_ptr()),
                    reinterpret_cast<uint8_t *>(y.data_ptr()),
                    reinterpret_cast<uint8_t *>(z.mutable_data_ptr()));
    return z;
}
}  // namespace ascendc_ops

PYBIND11_MODULE(custom_ops_lib, m)
{
    m.def("run_add_custom", &ascendc_ops::run_add_custom, "run add_custom Ascend C kernel");
}


`torch.library` 注册文件会写入 `source/06.04_super_kernel/custom_launch_sk/`。其中，Meta 实现用于 shape、dtype 等静态信息推导；`PrivateUse1` 实现用于 NPU 设备上的真实执行。

In [ ]:
%%writefile source/06.04_super_kernel/custom_launch_sk/op_extension.py
import torch

import custom_ops_lib


_LIBRARY = torch.library.Library("ascendc_ops", "FRAGMENT")
_LIBRARY.define("add_custom(Tensor x, Tensor y) -> Tensor")


@torch.library.impl(_LIBRARY, "add_custom", "Meta")
def add_custom_meta(x, y):
    return torch.empty_like(x)


@torch.library.impl(_LIBRARY, "add_custom", "PrivateUse1")
def add_custom_impl(x, y):
    return custom_ops_lib.run_add_custom(x, y)


构建配置已预置在 `src/06.04_super_kernel/custom_launch_sk/CMakeLists.txt`，执行以下单元将其复制到示例目录。


In [ ]:
%%bash
cp src/06.04_super_kernel/custom_launch_sk/CMakeLists.txt \
   source/06.04_super_kernel/custom_launch_sk/CMakeLists.txt


#### 2.3 示例：自定义 `<<<>>>` 算子入图并使能 SuperKernel

注册完成后，需要先导入扩展包触发 pybind 加载和 `torch.library` 注册，然后在模型中通过 `torch.ops.<namespace>.<op_name>` 调用自定义算子。

ACLGraph+SK 运行脚本会写入 `source/06.04_super_kernel/custom_launch_sk/`。扩展编译完成后，该脚本通过 `npugraph_ex` 捕获包含自定义 `<<<>>>` 算子的模型，并开启静态 Kernel 编译和 SuperKernel 融合。


In [ ]:
%%writefile source/06.04_super_kernel/custom_launch_sk/run_aclgraph_sk.py
import os

import torch
import torch_npu

import op_extension  # noqa: F401


class CustomAddModel(torch.nn.Module):
    def forward(self, x, y):
        torch.npu.super_kernel_scope_begin("custom_add_sk")
        z = torch.ops.ascendc_ops.add_custom(x, y)
        torch.npu.super_kernel_scope_end("custom_add_sk")
        return z


def main():
    device_id = int(os.environ.get("ASCEND_DEVICE_ID", os.environ.get("NPU_DEVICE_ID", "0")))
    torch.npu.set_device(device_id)

    shape = (8, 2048)
    x = torch.full(shape, 0.1, device="cpu", dtype=torch.float32).npu()
    y = torch.full(shape, 0.2, device="cpu", dtype=torch.float32).npu()

    model_sk = torch.compile(
        CustomAddModel(),
        backend="npugraph_ex",
        fullgraph=True,
        dynamic=False,
        options={
            "clone_input": False,
            "static_kernel_compile": True,
            "super_kernel_optimize": True,
        },
    )

    output = model_sk(x, y)
    torch.npu.synchronize()

    output_cpu = output.detach().cpu()
    output_sample = [round(value, 4) for value in output_cpu[0, :8].tolist()]
    print("input_x = 0.1")
    print("input_y = 0.2")
    print("expected_each = 0.3")
    print("graph_output_shape =", tuple(output_cpu.shape))
    print("graph_output[0, :8] =", output_sample)
    print("自定义 <<<>>> 算子 ACLGraph SuperKernel 示例运行完成。")
    return output


if __name__ == "__main__":
    main()


#### 2.4 编译并运行示例

执行以下脚本，完成自定义算子编译并运行 ACLGraph+SK 示例。示例使用固定输入 0.1 和 0.2，打印输出形状、结果片段和单元素预期值。


In [ ]:
%%bash
bash src/06.04_super_kernel/custom_launch_sk/build_and_run.sh


运行完成后，终端输出中的 `expected_each = 0.3` 是单个元素的预期结果，`graph_output[0, :8]` 是编译图执行后的结果片段。

## 3. ACLGraph 模式使能 SuperKernel

确认算子具备 SuperKernel 适配能力后，还需要在模型侧通过 npugraph_ex 将模型捕获为 ACLGraph，并开启 SuperKernel 融合优化。开启后，系统会分析 ACLGraph 捕获的 Model，识别其中可融合的连续 Task，将它们合并为一个 SuperKernel Task，并更新 Model 后执行。

### 3.1 通过 options 开启 SuperKernel 融合优化

使用 `torch.compile` 编译模型时，需要在 `options` 中同时开启静态 Kernel 编译和 SuperKernel 融合优化：

```python
options = {
    "static_kernel_compile": True,
    "super_kernel_optimize": True,
}

model_super_kernel = torch.compile(
    model,
    backend="npugraph_ex",
    options=options,
    dynamic=False,
    fullgraph=True,
)
```

使用时还需要注意：

- 开启后，算子 Data Dump 功能会失效。
- 当前支持参与融合的通信类算子包括 `AllReduce`、`ReduceScatter`、`AllGather` 和 `AlltoAll`。

### 3.2 标定 SuperKernel 融合范围

范围标定是可选操作。未标定时，系统会自动识别图内可融合的算子；需要控制融合范围时，可以使用一对 `super_kernel_scope_begin` 和 `super_kernel_scope_end`：

```python
torch.npu.super_kernel_scope_begin("sp1")
x = op1(x)
x = op2(x)
torch.npu.super_kernel_scope_end("sp1")
```

开始和结束位置需要使用相同的 `scope_name`，标定范围内的算子对应打上标记；传入 `None` 表示范围内的算子不参与 SuperKernel 融合。

当多个范围相交或嵌套时，系统会按照算子携带的 scope 标记集合进行分组，具体标记情况参考 [SuperKernel 融合优化功能](https://gitcode.com/Ascend/torchair/blob/master/docs/zh/npugraph_ex/advanced/superkernel.md)；`None` 标记的优先级最高。标定范围表达的是期望的融合范围，最终仍由 SuperKernel 融合规则决定实际结果。

### 3.3 使用示例

示例模型在 scope 内调用两个算子，并通过 `npugraph_ex` 的 `options` 开启静态 Kernel 编译和 SuperKernel 融合优化。代码会用 Python 脚本完成模型编译和执行。

In [ ]:
%%bash
set -eo pipefail

PYTHON_BIN=${PYTHON_BIN:-python3}

"${PYTHON_BIN}" - <<'PY'
import torch
import torch_npu
import os
device_id = int(os.environ.get("ASCEND_DEVICE_ID", "1"))
torch.npu.set_device(device_id)

class ModelSuperKernel(torch.nn.Module):
    def __init__(self, top_k):
        super().__init__()
        self.top_k = top_k

    def forward(self, gating_input, smooth_scales):
        torch.npu.super_kernel_scope_begin("sp1")
        gating_res = torch_npu.npu_moe_gating_top_k_softmax(
            gating_input, None, self.top_k
        )
        quant_res = torch_npu.npu_dynamic_quant(
            gating_res[0], smooth_scales=smooth_scales
        )
        torch.npu.super_kernel_scope_end("sp1")
        return quant_res, gating_res


token_num = 864
expert_num = 2048
top_k = 1024
gating_input = torch.randn((token_num, expert_num), dtype=torch.bfloat16)
smooth_scales = torch.randn((top_k,), dtype=torch.bfloat16)

model_super_kernel = torch.compile(
    ModelSuperKernel(top_k),
    backend="npugraph_ex",
    options={
        "static_kernel_compile": True,
        "super_kernel_optimize": True,
    },
    dynamic=False,
    fullgraph=True,
)
quant_res, gating_res = model_super_kernel(
    gating_input.npu(), smooth_scales.npu()
)
torch.npu.synchronize()
print("ACLGraph SuperKernel 示例运行完成。")
PY


### 3.4 options 使用方式

ACLGraph 模式的 SuperKernel 选项通过 `torch.compile` 的 `options` 传入。使能 SuperKernel 开关直接放在 `options` 顶层；融合优化类参数放在 `super_kernel_optimize_options` 字典中；调试类参数放在 `super_kernel_debug_options` 字典中。

```python
options = {
    "static_kernel_compile": True,
    "super_kernel_optimize": True,
    "super_kernel_optimize_options": {
        "early_start": 1,
        "dcci_before_kernel_start": [".*GroupedMatmul.*"],
    },
    "super_kernel_debug_options": {
        "debug_sync_all": 1,
    },
}
```

`super_kernel_debug_options` 中的选项会影响融合效果或运行性能，仅建议在定位问题时临时开启。

---

## 小结

- SuperKernel 通过算子二进制融合，减少任务调度、启动和部分 Cache Flush 开销，从而提升执行性能。
- ACLGraph 模式下，SuperKernel 通过编译选项开启，并支持按需标定融合范围；实际性能收益可通过 Profiling 对比分析。
- 自定义 `<<<>>>` 算子通过实现 `__sk__` 子函数并使用 `SK_BIND` 绑定，参与 SuperKernel 融合，同时需满足核数、同步和 Cache 一致性等适配约束。


---

## 4. 课后练习

1. （判断题）SuperKernel 会改变各子 Kernel 的计算语义，以减少模型的数学计算量。

2. （判断题）在 ACLGraph 模式下，只要设置 `super_kernel_optimize=True`，无需开启静态 Kernel 编译也可以使能 SuperKernel。

3. （单选题）在 ACLGraph 模式下，以下哪组 `torch.compile` 配置用于使能 SuperKernel？  
   A. `static_kernel_compile=True`，`super_kernel_optimize=True`  
   B. `static_kernel_compile=False`，`super_kernel_optimize=True`  
   C. `static_kernel_compile=True`，`super_kernel_debug_options=True`  
   D. `dynamic=True`，`super_kernel_optimize=False`

4. （单选题）需要标定一段连续算子的 SuperKernel 融合范围时，应如何操作？  
   A. 为每个算子设置不同的 `scope_name`  
   B. 使用相同 `scope_name` 的 `super_kernel_scope_begin/end` 包围该范围  
   C. 仅在范围末尾调用 `super_kernel_scope_end`  
   D. 将 `scope_name` 设置为 `None`，使范围内算子参与融合

5. （单选题）自定义 `<<<>>>` 算子已经通过 pybind 和 `torch.library` 接入 PyTorch，并能被 ACLGraph 捕获。若希望它作为 SuperKernel 内部子函数参与融合，还需要完成什么工作？  
   A. 仅增加 Python 侧的 `torch.compile` 调用  
   B. 将普通 Kernel 的 `<<<>>>` 启动核数固定为 1  
   C. 实现 `__sk__` 子函数，并通过 `SK_BIND` 与普通 Kernel 入口绑定  
   D. 删除普通 Kernel 入口，仅保留 `__sk__` 子函数

6. （多选题）关于本节基于 Profiling 数据进行的设备侧性能分析，下列说法正确的是：  
   A. `op_summary_*.csv` 记录设备侧任务的开始时间和执行时长  
   B. 连续任务区间的设备侧耗时可按“末任务开始时间 - 首任务开始时间 + 末任务执行时长”计算  
   C. 融合成功后，`op_summary_*.csv` 中可将该融合任务显示为 `SuperKernel` 类型  
   D. Host MSTX Range 总耗时可直接替代指定层的设备侧耗时

**执行以下代码获取答案。**


In [ ]:
!cat ./answer/06.04_answer.txt
